# Phase 1 Post-Formula-Decision Governance Update

Claim-closed orchestration notebook. This notebook links the final governance reconciliation state with the reviewed metric-formula decision. It does not rerun controls, calibration, influence, reporting, or model comparators; it only records the updated governance claim boundary.

Scientific-integrity rule: a formula decision of `unresolved` remains a blocker. Do not use this notebook to repair controls, reinterpret prior artifacts, or open claims.

In [ ]:
# Cell 1 - Bootstrap repo and run unit tests.

from pathlib import Path
import base64
import getpass
import hashlib
import json
import subprocess
from datetime import datetime, timezone

try:
    from google.colab import drive  # type: ignore
    drive.mount('/content/drive')
except Exception:
    pass

REPO_URL = 'https://github.com/BrianNguyen29/eeg-ds004752.git'
REPO_DIR = Path('/content/eeg-ds004752')
DRIVE_ROOT = Path('/content/drive/MyDrive/eeg-ds004752')
ARTIFACT_ROOT = DRIVE_ROOT / 'artifacts'
RUN_UNITTESTS = True

def run(cmd, cwd=None, check=True):
    printable = ' '.join(str(part) for part in cmd)
    if cmd and cmd[0] == 'git':
        printable = printable.replace(str(REPO_URL), 'https://github.com/BrianNguyen29/eeg-ds004752.git')
    print('$', printable)
    return subprocess.run(cmd, cwd=cwd, check=check, text=True)

if not REPO_DIR.exists():
    token = getpass.getpass('Nhap GitHub token cho repo private: ')
    auth = base64.b64encode(f'x-access-token:{token}'.encode()).decode()
    run(['git', '-c', f'http.extraHeader=Authorization: Basic {auth}', 'clone', REPO_URL, str(REPO_DIR)])
else:
    print('Repo exists:', REPO_DIR)
    run(['git', 'pull', '--ff-only'], cwd=REPO_DIR)

print('Repo:', REPO_DIR)
run(['git', 'log', '--oneline', '-5'], cwd=REPO_DIR, check=False)

if RUN_UNITTESTS:
    run(['python', '-m', 'unittest', 'tests.unit.test_phase1_final_post_formula_decision_governance_update'], cwd=REPO_DIR)


In [ ]:
# Cell 2 - Select reviewed sources and keep launch disabled by default.

EXPECTED_PREREG_IDENTITY_HASH = '87e928ea747099c336a32121bc156655a1a160d666a251c7ac41228efba96af6'
PREREG_BUNDLE = ARTIFACT_ROOT / 'prereg/20260418T161442014597Z/prereg_bundle.json'

# Use None to resolve latest.txt, or pin already reviewed runs.
PINNED_FINAL_GOVERNANCE_RECONCILIATION_RUN = None
PINNED_FORMULA_DECISION_RUN = None

OUTPUT_ROOT = ARTIFACT_ROOT / 'phase1_final_post_formula_decision_governance_update'
PLAN_ROOT = ARTIFACT_ROOT / 'phase1_final_post_formula_decision_governance_update_manual_hold'

RUN_POST_FORMULA_DECISION_GOVERNANCE_UPDATE = False
REQUIRED_ACK = 'I reviewed the post-formula-decision governance update and understand it reruns nothing changes no artifacts and opens no claims'
MANUAL_LAUNCH_ACK = ''
FIX_MARKER = 'phase1_final_post_formula_decision_governance_update_v1_20260423'
print('Notebook fix marker:', FIX_MARKER)


def resolve_latest(root: Path, required_file: str) -> Path:
    latest = root / 'latest.txt'
    if latest.exists():
        candidate = Path(latest.read_text().strip())
        if (candidate / required_file).exists():
            return candidate
    runs = sorted(path for path in root.iterdir() if path.is_dir()) if root.exists() else []
    for candidate in reversed(runs):
        if (candidate / required_file).exists():
            return candidate
    raise FileNotFoundError(f'No run containing {required_file} under {root}')

FINAL_GOVERNANCE_RECONCILIATION_RUN = Path(PINNED_FINAL_GOVERNANCE_RECONCILIATION_RUN) if PINNED_FINAL_GOVERNANCE_RECONCILIATION_RUN else resolve_latest(
    ARTIFACT_ROOT / 'phase1_final_governance_reconciliation',
    'phase1_final_governance_reconciliation_summary.json',
)
FORMULA_DECISION_RUN = Path(PINNED_FORMULA_DECISION_RUN) if PINNED_FORMULA_DECISION_RUN else resolve_latest(
    ARTIFACT_ROOT / 'phase1_final_controls_metric_formula_decision',
    'phase1_final_controls_metric_formula_decision_summary.json',
)

print('Final governance reconciliation run:', FINAL_GOVERNANCE_RECONCILIATION_RUN)
print('Formula decision run:', FORMULA_DECISION_RUN)
print('Run flag:', RUN_POST_FORMULA_DECISION_GOVERNANCE_UPDATE)


In [ ]:
# Cell 3 - Validate prereg, governance and formula-decision source boundaries.

def read_json(path: Path):
    return json.loads(path.read_text())

def sha256(path: Path):
    h = hashlib.sha256()
    with path.open('rb') as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b''):
            h.update(chunk)
    return h.hexdigest()

bundle = read_json(PREREG_BUNDLE)
actual_prereg_hash = bundle.get('prereg_bundle_hash_sha256')
raw_prereg_sha256 = sha256(PREREG_BUNDLE)
print('Prereg status:', bundle.get('status'))
print('Locked prereg identity hash:', actual_prereg_hash)
print('Raw prereg file SHA256:', raw_prereg_sha256)
assert bundle.get('status') == 'locked', 'Prereg bundle must be locked.'
assert actual_prereg_hash == EXPECTED_PREREG_IDENTITY_HASH, 'Locked prereg identity hash mismatch.'

gov_summary = read_json(FINAL_GOVERNANCE_RECONCILIATION_RUN / 'phase1_final_governance_reconciliation_summary.json')
gov_claim_state = read_json(FINAL_GOVERNANCE_RECONCILIATION_RUN / 'phase1_final_governance_claim_state.json')
formula_summary = read_json(FORMULA_DECISION_RUN / 'phase1_final_controls_metric_formula_decision_summary.json')
formula_claim_state = read_json(FORMULA_DECISION_RUN / 'phase1_final_controls_metric_formula_decision_claim_state.json')

assert gov_summary.get('claim_ready') is False
assert gov_summary.get('headline_phase1_claim_open') is False
assert gov_claim_state.get('headline_phase1_claim_open') is False
assert formula_summary.get('status') == 'phase1_final_controls_metric_formula_decision_recorded'
assert formula_summary.get('claim_ready') is False
assert formula_summary.get('claims_opened') is False
assert formula_summary.get('thresholds_changed') is False
assert formula_summary.get('controls_rerun_by_decision_runner') is False
assert formula_summary.get('logits_or_metrics_edited') is False
assert formula_claim_state.get('headline_phase1_claim_open') is False

print(json.dumps({
    'governance_run': str(FINAL_GOVERNANCE_RECONCILIATION_RUN),
    'governance_surfaces': gov_summary.get('governance_surfaces'),
    'formula_decision_run': str(FORMULA_DECISION_RUN),
    'formula_decision': formula_summary.get('formula_decision'),
    'selected_formula': formula_summary.get('selected_formula'),
    'formula_next_step': formula_summary.get('next_step'),
}, indent=2))


In [ ]:
# Cell 4 - Record manual-hold plan and command preview.

PLAN_ROOT.mkdir(parents=True, exist_ok=True)
plan_dir = PLAN_ROOT / datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
plan_dir.mkdir(parents=True, exist_ok=False)

runner_available = (REPO_DIR / 'src/phase1/final_post_formula_decision_governance_update.py').exists()
preflight_blockers = []
if not runner_available:
    preflight_blockers.append('src/phase1/final_post_formula_decision_governance_update.py missing')

cmd = [
    'python', '-m', 'src.cli', 'phase1_final_post_formula_decision_governance_update',
    '--profile', 't4_safe',
    '--config', str(PREREG_BUNDLE),
    '--final-governance-reconciliation-run', str(FINAL_GOVERNANCE_RECONCILIATION_RUN),
    '--formula-decision-run', str(FORMULA_DECISION_RUN),
    '--output-root', str(OUTPUT_ROOT),
]

plan = {
    'status': 'phase1_final_post_formula_decision_governance_update_manual_hold',
    'created_utc': plan_dir.name,
    'runner_available': runner_available,
    'run_flag': RUN_POST_FORMULA_DECISION_GOVERNANCE_UPDATE,
    'required_ack': REQUIRED_ACK,
    'ack_matched': MANUAL_LAUNCH_ACK == REQUIRED_ACK,
    'final_governance_reconciliation_run': str(FINAL_GOVERNANCE_RECONCILIATION_RUN),
    'formula_decision_run': str(FORMULA_DECISION_RUN),
    'formula_decision': formula_summary.get('formula_decision'),
    'preflight_blockers': preflight_blockers,
    'command': cmd,
    'scientific_integrity_rule': 'This update links governance and formula-decision artifacts only; it reruns nothing and opens no claims.',
}
(plan_dir / 'phase1_final_post_formula_decision_governance_update_manual_hold.json').write_text(json.dumps(plan, indent=2) + '
')
print(json.dumps(plan, indent=2))


In [ ]:
# Cell 5 - Manual launch gate.

if preflight_blockers:
    raise RuntimeError(f'Preflight blockers must be resolved before governance update: {preflight_blockers}')
elif not RUN_POST_FORMULA_DECISION_GOVERNANCE_UPDATE:
    print('HELD: Post-formula-decision governance update was not launched because manual flag is False.')
    print('NEXT: review the source artifacts, then rerun only with explicit claim-closed acknowledgement if appropriate.')
elif MANUAL_LAUNCH_ACK != REQUIRED_ACK:
    raise RuntimeError('Manual launch acknowledgement mismatch. Do not record governance update without explicit claim-closed acknowledgement.')
else:
    run(cmd, cwd=REPO_DIR)
    print('Post-formula-decision governance update completed. Review generated artifacts before any remediation planning.')


In [ ]:
# Cell 6 - Inspect latest output if launched.

latest_output = None
if RUN_POST_FORMULA_DECISION_GOVERNANCE_UPDATE and MANUAL_LAUNCH_ACK == REQUIRED_ACK:
    latest_output = resolve_latest(OUTPUT_ROOT, 'phase1_final_post_formula_decision_governance_update_summary.json')
    summary = read_json(latest_output / 'phase1_final_post_formula_decision_governance_update_summary.json')
    metric_formula_status = read_json(latest_output / 'phase1_final_metric_formula_contract_status.json')
    claim_state = read_json(latest_output / 'phase1_final_post_formula_decision_governance_claim_state.json')

    print('Latest post-formula governance output:', latest_output)
    print('Formula decision:', summary.get('formula_decision'))
    print('Selected formula:', summary.get('selected_formula'))
    print('Metric formula claim-evaluable:', summary.get('metric_formula_contract_claim_evaluable'))
    print('Metric formula next step:', summary.get('metric_formula_next_step'))
    print('Claims opened:', summary.get('claims_opened'))
    print('Claim blockers:', claim_state.get('blockers'))
else:
    print('No governance update output inspected because launch is held.')


In [ ]:
# Cell 7 - Assertions and review note.

if latest_output is not None:
    assert summary.get('status') == 'phase1_final_post_formula_decision_governance_update_recorded', summary.get('claim_blockers')
    assert summary.get('claim_ready') is False
    assert summary.get('headline_phase1_claim_open') is False
    assert summary.get('claims_opened') is False
    assert metric_formula_status.get('claim_evaluable') is False
    assert claim_state.get('headline_phase1_claim_open') is False
    assert any(str(blocker).startswith('metric_formula_contract:') for blocker in claim_state.get('blockers', []))

    review_note = {
        'status': 'phase1_final_post_formula_decision_governance_update_review_pass_claim_closed',
        'reviewed_run': str(latest_output),
        'formula_decision': summary.get('formula_decision'),
        'selected_formula': summary.get('selected_formula'),
        'metric_formula_contract_claim_evaluable': summary.get('metric_formula_contract_claim_evaluable'),
        'checks_passed': [
            'governance_update_recorded',
            'claim_ready_false',
            'headline_claim_open_false',
            'claims_opened_false',
            'metric_formula_contract_not_claim_evaluable',
            'metric_formula_blocker_preserved',
        ],
        'next_step': summary.get('metric_formula_next_step'),
        'not_ok_to_claim': claim_state.get('not_ok_to_claim'),
    }
    note_path = latest_output / 'phase1_final_post_formula_decision_governance_update_review_note.json'
    note_path.write_text(json.dumps(review_note, indent=2) + '
')
    print('Review note written:', note_path)
    print(json.dumps(review_note, indent=2))
else:
    print('Assertions skipped because launch is held.')


In [ ]:
# Cell 8 - Closeout message.

print('================ Phase 1 Post-Formula-Decision Governance Update Closeout ================')
print('Notebook fix marker:', FIX_MARKER)
print('Plan source of truth:', plan_dir)
print('Runner available:', runner_available)
print('Run flag:', RUN_POST_FORMULA_DECISION_GOVERNANCE_UPDATE)
print('Final governance reconciliation run:', FINAL_GOVERNANCE_RECONCILIATION_RUN)
print('Formula decision run:', FORMULA_DECISION_RUN)
print('Formula decision:', formula_summary.get('formula_decision'))
print('Expected locked prereg identity hash:', EXPECTED_PREREG_IDENTITY_HASH)
print('Locked prereg hash from bundle:', actual_prereg_hash)
print('Raw prereg file SHA256:', raw_prereg_sha256)

if latest_output is None:
    print('HELD: Governance update was not launched because manual flag is False.')
    print('NEXT: rerun only with explicit claim-closed acknowledgement if the reviewed sources are correct.')
else:
    print('Latest post-formula governance output:', latest_output)
    print('Metric formula claim-evaluable:', summary.get('metric_formula_contract_claim_evaluable'))
    print('Metric formula next step:', summary.get('metric_formula_next_step'))
    print('Claims opened:', summary.get('claims_opened'))
    print('CHECK REQUIRED: Review phase1_final_post_formula_decision_governance_decision_memo.md before any future remediation planning.')

print('NOT OK TO CLAIM: This notebook does not prove decoder efficacy, A2d efficacy, A3/A4 efficacy, A4 superiority, privileged-transfer efficacy, or full Phase 1 neural comparator performance.')
